# Exact Ω Computation
**목적:** 각 레이어에 대해 `Δθ_i^T H_ii Δθ_i` (exact block-diagonal Hessian quadratic form)을 계산

현재 CSV에 있는 `Tr(H_ii) × ‖Δθ_i‖²` (trace approximation)와 비교하기 위함.

**방법:** Hessian-vector product (HVP) via double-backprop
```
g = ∂L/∂θ_i  (with create_graph=True)
Hv = ∂(g·v)/∂θ_i  where v = Δθ_i
exact_Ω_i = Δθ_i · Hv
```

**출력:** `results/csv/hawq_exact_omega.csv`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os

BASE = '/content/drive/MyDrive/MambaCompression/MambaIC'
sys.path.insert(0, BASE)
os.chdir(BASE)
print('CWD:', os.getcwd())

In [ ]:
import shutil

CACHE_DIR = os.path.normpath(os.path.join(BASE, '..', '_kernel_cache'))
SO_NAME   = 'selective_scan_cuda_oflex.cpython-312-x86_64-linux-gnu.so'
SITE_PKG  = '/usr/local/lib/python3.12/dist-packages'

def _try_import():
    try:
        import selective_scan_cuda_oflex
        return True
    except Exception:
        return False

if _try_import():
    print('selective_scan_cuda_oflex: already installed')
else:
    so_src = os.path.join(CACHE_DIR, SO_NAME)
    so_dst = os.path.join(SITE_PKG, SO_NAME)
    if os.path.exists(so_src):
        shutil.copy2(so_src, so_dst)
        if not _try_import():
            os.remove(so_dst)
            os.system('pip install mamba-ssm causal-conv1d -q')
        else:
            print('loaded from cache')
    else:
        os.system('pip install mamba-ssm causal-conv1d -q')
        so_built = os.path.join(SITE_PKG, SO_NAME)
        if os.path.exists(so_built):
            os.makedirs(CACHE_DIR, exist_ok=True)
            shutil.copy2(so_built, os.path.join(CACHE_DIR, SO_NAME))

import models.VSS_module
try:
    import selective_scan_cuda_oflex
    print('VSS module: OK')
except ImportError:
    print('VSS module: CUDA kernel missing')

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import pandas as pd
import scipy.io as sio
from ModularModels import ModularAE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
MODEL_PATH = 'saved_models/mamba_transnet_L2_dim512_baseline/best.pth'

model = ModularAE(
    encoder_type='mamba',
    decoder_type='transnet',
    encoded_dim=512,
    decoder_layers=2
)
ck = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(ck['state_dict'], strict=False)
model = model.to(device).eval()
print(f'Model loaded (epoch={ck["epoch"]})')

In [ ]:
DATA_PATH  = 'data/DATA_Htestout.mat'
BATCH_SIZE = 32
N_BATCHES  = 10

mat   = sio.loadmat(DATA_PATH)
H_raw = mat['HT'].astype('float32')

if H_raw.ndim == 2:
    H_raw = H_raw.reshape(-1, 2, 32, 32)

H_raw = H_raw - H_raw.mean()  # placeholder

x_all   = torch.FloatTensor(H_raw)
dataset = torch.utils.data.TensorDataset(x_all[:N_BATCHES * BATCH_SIZE])
loader  = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False)
print(f'Calibration: {len(dataset)} samples')

In [ ]:
def quantize(w, bits):
    if bits >= 16:
        return w.clone()
    qmax  = 2 ** (bits - 1) - 1
    scale = w.abs().max().clamp(min=1e-8) / qmax
    return (w / scale).round().clamp(-qmax, qmax) * scale

def delta(w, bits):
    return (quantize(w, bits) - w).detach()

print('Quantize functions defined')

In [ ]:
FC_CHUNK = 16

LAYER_MAP = {
    'stem.0':                  ('encoder.stem.0.weight',              None),
    'layers.0.vss.1.in_proj':  ('encoder.layers.0.vss.1.in_proj.weight',  None),
    'layers.0.vss.1.out_proj': ('encoder.layers.0.vss.1.out_proj.weight', None),
    'layers.0.vss.1.conv2d':   ('encoder.layers.0.vss.1.conv2d.weight',   None),
    'layers.1.vss.1.in_proj':  ('encoder.layers.1.vss.1.in_proj.weight',  None),
    'layers.1.vss.1.out_proj': ('encoder.layers.1.vss.1.out_proj.weight', None),
    'layers.1.vss.1.conv2d':   ('encoder.layers.1.vss.1.conv2d.weight',   None),
    'proj_conv':               ('encoder.proj_conv.weight',               None),
}
for k in range(32):
    LAYER_MAP[f'fc_part{k}'] = ('encoder.fc.weight', (k * FC_CHUNK, (k + 1) * FC_CHUNK))

param_dict = dict(model.named_parameters())
BITS = [16, 8, 4, 2]
print(f'{len(LAYER_MAP)} layers to process')

In [ ]:
omega_acc = {ln: {b: 0.0 for b in BITS} for ln in LAYER_MAP}
n_done = 0

for ln, (pname, chunk) in LAYER_MAP.items():
    param = param_dict[pname]

    for bits in BITS:
        if chunk is None:
            dtheta = delta(param, bits).to(device)
            v = dtheta
        else:
            r0, r1 = chunk
            dtheta = delta(param[r0:r1, :], bits).to(device)
            v = torch.zeros_like(param)
            v[r0:r1, :] = dtheta

        if dtheta.abs().max() < 1e-10:
            omega_acc[ln][bits] = 0.0
            continue

        batch_sum, batch_cnt = 0.0, 0
        for b_idx, (x,) in enumerate(loader):
            if b_idx >= N_BATCHES:
                break
            x = x.to(device)
            with torch.enable_grad():
                recon = model(x)
                loss  = F.mse_loss(recon, x)
            g  = torch.autograd.grad(loss, param, create_graph=True)[0]
            Hv = torch.autograd.grad((g * v).sum(), param, retain_graph=False)[0]
            if chunk is None:
                batch_sum += (v * Hv).sum().item()
            else:
                batch_sum += (dtheta * Hv[r0:r1, :]).sum().item()
            batch_cnt += 1
            del loss, recon, g, Hv
            torch.cuda.empty_cache()

        omega_acc[ln][bits] = batch_sum / max(batch_cnt, 1)

    n_done += 1
    print(f'[{n_done:3d}/{len(LAYER_MAP)}] {ln}')

print('Done')

In [ ]:
rows = []
for ln, (pname, chunk) in LAYER_MAP.items():
    param = param_dict[pname]
    n_p   = param[chunk[0]:chunk[1], :].numel() if chunk else param.numel()
    rows.append({
        'Layer':          ln,
        'Params':         n_p,
        'ExactOmg_INT16': omega_acc[ln][16],
        'ExactOmg_INT8':  omega_acc[ln][8],
        'ExactOmg_INT4':  omega_acc[ln][4],
        'ExactOmg_INT2':  omega_acc[ln][2],
    })

df_exact = pd.DataFrame(rows)
df_trace = pd.read_csv('results/csv/hawq_importance_split.csv')
df_cmp   = df_exact.merge(
    df_trace[['Layer', 'Omg_INT8', 'Omg_INT4', 'Omg_INT2']], on='Layer', how='left')
df_cmp['Ratio_INT8'] = df_cmp['ExactOmg_INT8'] / df_cmp['Omg_INT8'].clip(lower=1e-15)
df_cmp['Ratio_INT4'] = df_cmp['ExactOmg_INT4'] / df_cmp['Omg_INT4'].clip(lower=1e-15)

out_path = 'results/csv/hawq_exact_omega.csv'
df_cmp.to_csv(out_path, index=False)
print(f'Saved: {out_path}')
print(df_cmp[['Layer','ExactOmg_INT8','Omg_INT8','Ratio_INT8']].head(15).to_string())